# 🕵️ The Secret Message Game: Steganography and Frequency Analysis

## Welcome, Cryptographer!

What if I told you that an innocent-looking image could contain a secret message? Not just hidden *inside* it, but *as* the actual pixels themselves?

In this notebook, you'll learn:
- How colors can encode letters
- How to hide messages in images (steganography)
- How to crack codes using frequency analysis (like a real cryptographer!)
- How to visualize data with matplotlib

Best of all? You'll see why those "matrix things" you've been learning are actually **super useful**!

---

## The Big Idea

Computer colors use **24-bit RGB** (Red, Green, Blue). Each color channel can be 0-255, giving us:
- 256 × 256 × 256 = **16,777,216 different colors**

But the English alphabet only has 26 letters, plus some punctuation. That means we have WAY more colors than we need to encode text!

We can create an image that looks like random colors... but is actually a secret message! 🤫

## Setup: Import Our Tools

Let's import what we need. Don't worry if you haven't used these before - we'll explain as we go!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import random

# Make plots look nice
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All tools loaded! Ready to decode secrets...")

---

# 🎯 Level 1: Simple ASCII Steganography

Let's start with the basics. We'll encode each letter as a color using its ASCII value.

**ASCII** (American Standard Code for Information Interchange) is a way to represent text as numbers:
- 'A' = 65
- 'B' = 66
- 'Z' = 90
- 'a' = 97
- ' ' (space) = 32

We'll map each character to a unique color!

In [ ]:
def text_to_ascii_colors(message):
    """
    Convert text to colors using ASCII values.
    Each character becomes a unique color!
    """
    colors = []
    for char in message:
        # Get ASCII value (0-127 for standard ASCII)
        ascii_val = ord(char)
        
        # Create a color: use ASCII value in different channels
        # This creates distinct colors for each character
        red = (ascii_val * 3) % 256
        green = (ascii_val * 7) % 256
        blue = (ascii_val * 11) % 256
        
        # Normalize to 0-1 range (matplotlib uses 0-1 for colors)
        colors.append([red/255, green/255, blue/255])
    
    return colors

def create_image_from_colors(colors, width=None):
    """
    Turn a list of colors into a 2D image (like a matrix!).
    Each color becomes one pixel.
    """
    num_pixels = len(colors)
    
    # If no width specified, make it roughly square
    if width is None:
        width = int(num_pixels ** 0.5) + 1
    
    # Calculate height needed
    height = (num_pixels // width) + 1
    
    # Create our image matrix
    image = []
    for row in range(height):
        image_row = []
        for col in range(width):
            idx = row * width + col
            if idx < num_pixels:
                image_row.append(colors[idx])
            else:
                # Fill extra space with black
                image_row.append([0, 0, 0])
        image.append(image_row)
    
    return image

# Test it out!
secret = "HELLO WORLD"
print(f"Our secret message: '{secret}'")
print(f"Number of characters: {len(secret)}")
print(f"\nFirst character '{secret[0]}' has ASCII value: {ord(secret[0])}")

### Let's Create and View Our Secret Image!

In [ ]:
# Encode the message
colors = text_to_ascii_colors(secret)
image = create_image_from_colors(colors, width=6)

# Display it!
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(f"Secret Image (contains {len(secret)} characters)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("🎨 Above is an image with a secret message!")
print("Each colored square is one letter or character.")

### Now Let's Decode It!

We'll reverse the process to get our message back.

In [ ]:
def colors_to_ascii_text(colors):
    """
    Convert colors back to text using ASCII values.
    This is the REVERSE of our encoding!
    """
    message = ""
    
    for color in colors:
        # Convert back from 0-1 range to 0-255
        red = int(color[0] * 255)
        green = int(color[1] * 255)
        blue = int(color[2] * 255)
        
        # Try all possible ASCII values to find which one matches
        for ascii_val in range(128):
            test_r = (ascii_val * 3) % 256
            test_g = (ascii_val * 7) % 256
            test_b = (ascii_val * 11) % 256
            
            if test_r == red and test_g == green and test_b == blue:
                message += chr(ascii_val)
                break
    
    return message

# Decode our message!
decoded = colors_to_ascii_text(colors)
print(f"🔓 Decoded message: '{decoded}'")
print(f"\n✅ Success! We recovered the secret message!")

### 🎮 Your Turn: Try Your Own Message!

Encode and decode your own secret message:

In [ ]:
# Put your own message here!
my_secret = "I LOVE PYTHON"

# Encode it
my_colors = text_to_ascii_colors(my_secret)
my_image = create_image_from_colors(my_colors, width=8)

# Show it
plt.imshow(my_image)
plt.title("My Secret Image", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

# Decode it
recovered = colors_to_ascii_text(my_colors)
print(f"Decoded: '{recovered}'")

---

# 🔥 Level 2: Frequency Analysis - Breaking the Code!

Now for the REALLY cool part! What if someone scrambles the color-to-letter mapping?

## The Problem

Instead of 'A' → color1, 'B' → color2, etc., they use a **cipher**:
- 'E' → color17
- 'T' → color5
- 'A' → color23
- etc.

You can't just reverse the color formula anymore! 😱

## The Solution: Frequency Analysis!

Here's a cool fact: **in English, some letters appear WAY more often than others!**

Most common letters: **E, T, A, O, I, N, S, H, R**

If we:
1. Count letter frequencies in English text (like Shakespeare)
2. Count color frequencies in our secret image
3. Match them up

We can crack the code! This is how real cryptographers work!

## Step 1: Analyze English Letter Frequencies

Let's use a sample of English text to learn the typical letter frequencies.

In [ ]:
# A sample of English text (excerpt from a public domain text)
# This is from Shakespeare's works - notice how some letters appear more!
sample_text = """
To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them To die to sleep
No more and by a sleep to say we end
The heart ache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished To die to sleep
To sleep perchance to dream ay theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause theres the respect
That makes calamity of so long life
"""

# For better results, you can also use a longer text!
# Uncomment below to use a much larger sample:

longer_text = """
All the world is a stage and all the men and women merely players
They have their exits and their entrances And one man in his time plays many parts
His acts being seven ages At first the infant mewling and puking in the nurses arms
Then the whining schoolboy with his satchel and shining morning face
creeping like snail unwillingly to school And then the lover sighing like furnace
with a woeful ballad made to his mistress eyebrow Then a soldier full of strange oaths
and bearded like the pard jealous in honor sudden and quick in quarrel
seeking the bubble reputation even in the cannons mouth And then the justice
in fair round belly with good capon lined with eyes severe and beard of formal cut
full of wise saws and modern instances and so he plays his part
The sixth age shifts into the lean and slippered pantaloon with spectacles on nose
and pouch on side his youthful hose well saved a world too wide for his shrunk shank
and his big manly voice turning again toward childish treble pipes and whistles in his sound
Last scene of all that ends this strange eventful history is second childishness
and mere oblivion sans teeth sans eyes sans taste sans everything
""" * 5  # Repeat it for more data!

# Use the longer text for better frequency analysis
training_text = longer_text

In [ ]:
def analyze_letter_frequencies(text):
    """
    Count how often each letter appears in text.
    Returns a sorted list: most common letters first!
    """
    # Convert to uppercase and keep only letters and spaces
    clean_text = ""
    for char in text.upper():
        if char.isalpha() or char == ' ':
            clean_text += char
    
    # Count each letter
    letter_counts = Counter(clean_text)
    
    # Sort by frequency (most common first)
    sorted_letters = letter_counts.most_common()
    
    return sorted_letters

# Analyze our training text
english_freq = analyze_letter_frequencies(training_text)

print("📊 Letter Frequencies in English:")
print("\nTop 10 most common characters:")
for char, count in english_freq[:10]:
    percentage = (count / sum(c for _, c in english_freq)) * 100
    print(f"  '{char}': {count:4d} times ({percentage:5.2f}%)")

### Visualize the Frequencies!

In [ ]:
# Prepare data for plotting
letters = [char for char, _ in english_freq[:15]]
counts = [count for _, count in english_freq[:15]]

# Create bar chart
plt.figure(figsize=(14, 6))
bars = plt.bar(letters, counts, color='steelblue', edgecolor='navy', linewidth=1.5)

# Highlight the most common letter
bars[0].set_color('crimson')

plt.xlabel('Character', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title('Letter Frequencies in English Text', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 Notice: '{letters[0]}' is the most common character!")
print("This pattern is consistent across most English text.")

## Step 2: Create an Encrypted Message

Now let's create a secret message with a SCRAMBLED color mapping!

In [ ]:
def create_random_cipher():
    """
    Create a random color for each letter.
    This is our SECRET cipher that we'll try to crack!
    """
    cipher = {}
    used_colors = set()
    
    # All possible characters (uppercase letters + space)
    characters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ '
    
    for char in characters:
        # Generate a random, unique color
        while True:
            color = (random.randint(0, 255), 
                    random.randint(0, 255), 
                    random.randint(0, 255))
            if color not in used_colors:
                used_colors.add(color)
                break
        
        cipher[char] = color
    
    return cipher

def encode_with_cipher(message, cipher):
    """
    Encode a message using our cipher.
    """
    colors = []
    for char in message.upper():
        if char in cipher:
            color = cipher[char]
            # Normalize to 0-1
            colors.append([color[0]/255, color[1]/255, color[2]/255])
    return colors

# Create our secret cipher
random.seed(42)  # For reproducibility - remove for truly random!
secret_cipher = create_random_cipher()

# Our hidden message (a famous quote!)
hidden_message = "THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG"

print(f"📝 Hidden message length: {len(hidden_message)} characters")
print("🔐 Cipher created! The color-to-letter mapping is now scrambled!")
print("\nExample: In this cipher...")
example_char = 'E'
example_color = secret_cipher[example_char]
print(f"  Letter '{example_char}' → RGB{example_color}")

In [ ]:
# Encode the message
encrypted_colors = encode_with_cipher(hidden_message, secret_cipher)
encrypted_image = create_image_from_colors(encrypted_colors, width=10)

# Display the encrypted image
plt.figure(figsize=(10, 6))
plt.imshow(encrypted_image)
plt.title("🔒 Encrypted Image (Can you crack it?)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("Above is an image with a secret message!")
print("But this time, we used a RANDOM cipher.")
print("We can't just reverse the color formula...")
print("\n🤔 How do we crack it? FREQUENCY ANALYSIS! ⬇️")

## Step 3: Crack the Code!

Now the fun part - let's use frequency analysis to break the cipher!

In [ ]:
def analyze_color_frequencies(colors):
    """
    Count how often each color appears in our image.
    """
    # Convert colors to tuples so we can count them
    color_tuples = []
    for color in colors:
        # Convert back to RGB integers
        r = int(color[0] * 255)
        g = int(color[1] * 255)
        b = int(color[2] * 255)
        color_tuples.append((r, g, b))
    
    # Count frequencies
    color_counts = Counter(color_tuples)
    
    # Sort by frequency
    return color_counts.most_common()

# Analyze the encrypted image
color_freq = analyze_color_frequencies(encrypted_colors)

print("🎨 Color Frequencies in Encrypted Image:")
print("\nTop 10 most common colors:")
for color, count in color_freq[:10]:
    percentage = (count / len(encrypted_colors)) * 100
    print(f"  RGB{color}: {count:3d} times ({percentage:5.2f}%)")

### The Magic: Match Frequencies!

Here's the cryptographer's trick:
- Most common color → Most common letter (probably 'E' or space)
- 2nd most common color → 2nd most common letter
- And so on...

Let's create our decryption mapping!

In [ ]:
def crack_cipher(encrypted_colors, english_frequencies):
    """
    Use frequency analysis to crack the cipher!
    Match most common colors to most common letters.
    """
    # Get color frequencies from encrypted message
    color_freq = analyze_color_frequencies(encrypted_colors)
    
    # Create decryption mapping
    decryption_map = {}
    
    # Match them up!
    for i, (color, _) in enumerate(color_freq):
        if i < len(english_frequencies):
            # Most common color gets most common letter, etc.
            letter = english_frequencies[i][0]
            decryption_map[color] = letter
    
    return decryption_map

def decrypt_message(encrypted_colors, decryption_map):
    """
    Use our cracked cipher to decrypt the message!
    """
    message = ""
    
    for color in encrypted_colors:
        # Convert to RGB tuple
        r = int(color[0] * 255)
        g = int(color[1] * 255)
        b = int(color[2] * 255)
        rgb = (r, g, b)
        
        # Look up the letter
        if rgb in decryption_map:
            message += decryption_map[rgb]
        else:
            message += '?'  # Unknown
    
    return message

# CRACK THE CODE!
print("🔓 Attempting to crack the cipher...\n")

cracked_map = crack_cipher(encrypted_colors, english_freq)
decrypted_message = decrypt_message(encrypted_colors, cracked_map)

print("="*60)
print("📨 DECRYPTED MESSAGE:")
print("="*60)
print(f"\n{decrypted_message}\n")
print("="*60)
print(f"\n🎯 Original message: {hidden_message}")
print(f"🔍 Cracked message:  {decrypted_message}")

# Check accuracy
correct = sum(1 for i in range(len(hidden_message)) if decrypted_message[i] == hidden_message[i])
accuracy = (correct / len(hidden_message)) * 100
print(f"\n✨ Accuracy: {accuracy:.1f}%")

if accuracy > 90:
    print("🏆 Excellent! Frequency analysis works like magic!")
elif accuracy > 70:
    print("👍 Pretty good! Some letters might be wrong, but we got the gist!")
else:
    print("🤔 Hmm, we need more text to analyze for better results!")

### Visualize the Cipher Mapping

In [ ]:
# Show the mapping we discovered
print("🔑 Our Cracked Cipher (top 10):")
print("\nColor → Letter (our guess)")
print("-" * 40)

for i, (color, count) in enumerate(color_freq[:10]):
    if color in cracked_map:
        letter = cracked_map[color]
        print(f"RGB{color} → '{letter}' ({count} times)")
    if i >= 9:
        break

---

# 🎓 What You Learned

Congratulations! You just:

1. **Used matrices/arrays** to represent images (rows and columns of colors!)
2. **Encoded information** into images (steganography)
3. **Applied frequency analysis** to break a cipher (real cryptography!)
4. **Visualized data** with matplotlib
5. **Worked with colors as numbers** (RGB values)

## Why This Matters

- **Images are just matrices** of numbers - this is how computers "see"!
- **Frequency analysis** is used in real cryptography, data science, and even spell-checkers!
- **Steganography** (hiding data in images) is used in security, watermarking, and more
- **Thinking with matrices** is fundamental to image processing, machine learning, and graphics

---

# 🚀 Challenges: Take It Further!

Ready for more? Try these:

## Challenge 1: Bigger Message

Try encoding a longer message (maybe a whole paragraph!) and see if frequency analysis still works.

In [ ]:
# Your code here!
long_message = "YOUR LONGER MESSAGE HERE"

# Encode, display, and decrypt it!


## Challenge 2: Load Your Own Text

Find a text file (maybe a book from Project Gutenberg?) and use it for frequency analysis. Does it improve accuracy?

In [ ]:
# Example: Load a text file
# with open('your_text_file.txt', 'r') as f:
#     custom_text = f.read()
# 
# custom_freq = analyze_letter_frequencies(custom_text)
# # Then use custom_freq for decryption!


## Challenge 3: Add More Characters

Right now we only use uppercase letters and spaces. Can you extend it to include:
- Lowercase letters
- Punctuation (. , ! ?)
- Numbers

In [ ]:
# Hint: You'll need to modify the cipher creation and frequency analysis
# to include more characters!


## Challenge 4: Visualize the Cracking Process

Can you create a visualization showing:
- How many of each color appear
- How they map to letters
- Maybe even show the partial decryption process?

In [ ]:
# Create a side-by-side comparison:
# - Bar chart of color frequencies
# - Bar chart of letter frequencies  
# - Show how they match up!


## Challenge 5: Improve the Cracking Algorithm

Frequency analysis alone might not be perfect. Can you improve it by:
- Looking at common letter pairs (bigrams) like 'TH', 'HE', 'AN'
- Using dictionary words to verify guesses
- Implementing a "scoring" system for how "English-like" the result is

In [ ]:
# Advanced cryptography techniques!
# common_bigrams = ['TH', 'HE', 'IN', 'ER', 'AN', 'RE', 'ON', 'AT', 'EN', 'ND']


---

# 🎉 Congratulations!

You've completed the Secret Message Game! You now know:

- How steganography works
- How to use frequency analysis (a real cryptography technique!)
- How images are really just matrices of numbers
- How to visualize data with matplotlib
- That the math you're learning has real, practical (and fun!) applications!

## Want to Learn More?

- **Cryptography**: Look up Caesar cipher, Vigenère cipher, and modern encryption
- **Steganography**: Research LSB (Least Significant Bit) steganography
- **Image Processing**: Explore PIL/Pillow and OpenCV libraries
- **Data Analysis**: Learn about pandas, NumPy, and data science!

Keep coding, keep learning, and keep having fun! 🚀

---

*This notebook demonstrates practical applications of arrays/matrices, data analysis, and visualization - fundamental skills in programming, data science, and computer science!*